# Chatbot Professor Particular de Algoritmos com Qwen Coder

```python
MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct-GPTQ-Int4"
```

## 1. Instalação das bibliotecas

In [ ]:
!pip install -q --upgrade transformers accelerate sentencepiece pandas autoawq gptqmodel

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.3 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.2.6 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 7.35.1 which is incompatible.


In [ ]:
!pip uninstall -y numpy
!pip install -q "numpy<2.0.0" pandas --force-reinstall

Found existing installation: numpy 2.2.6
Uninstalling numpy-2.2.6:
  Successfully uninstalled numpy-2.2.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gptqmodel 7.1.0 requires numpy==2.2.6; python_version < "3.14", but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.3 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you ha

## 2. Importações e verificação do ambiente

In [ ]:
import gc
import time
import warnings
from datetime import datetime

import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM

warnings.filterwarnings("ignore")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo disponível: {DEVICE}")

if DEVICE == "cuda":
    print(f"GPU detectada: {torch.cuda.get_device_name(0)}")
    memoria_total = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"Memória total da GPU: {memoria_total:.2f} GB")
else:
    print("GPU não detectada. O modelo pode ficar muito lento ou não carregar em CPU.")

Dispositivo disponível: cuda
GPU detectada: Tesla T4
Memória total da GPU: 14.56 GB


## 3. Configuração do modelo

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct-AWQ"

print(f"Modelo selecionado: {MODEL_ID}")

Modelo selecionado: Qwen/Qwen2.5-Coder-7B-Instruct-AWQ


## 4. Carregamento do tokenizer e do modelo

In [ ]:
print("Carregando tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Carregando modelo...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype="auto",
    trust_remote_code=True
)

model.eval()

print("\nModelo e tokenizer carregados com sucesso!")
print(f"Tipo do modelo: {type(model)}")

try:
    print(f"Dispositivo principal do modelo: {model.device}")
except Exception:
    print("O modelo foi distribuído automaticamente pelo device_map='auto'.")

Carregando tokenizer...


Carregando modelo...
Atenção: a primeira execução pode parecer parada enquanto o modelo é baixado e preparado.


model.safetensors.index.json:   0%|          | 0.00/62.7k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.


INFO  ENV: Auto setting PYTORCH_ALLOC_CONF='expandable_segments:True,max_split_size_mb:256,garbage_collection_threshold:0.7' for memory saving.


... (6 outputs left out)

Loading weights:   0%|          | 0/731 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

INFO  gc.collect() reclaimed 147 objects in 0.313s                             



Modelo e tokenizer carregados com sucesso!
Tipo do modelo: <class 'transformers.models.qwen2.modeling_qwen2.Qwen2ForCausalLM'>
Dispositivo principal do modelo: cuda:0


## 5. Prompt de sistema: Professor de Algoritmos

Aqui definimos o comportamento do chatbot.

A especialização inicial é feita por engenharia de prompt. O modelo recebe instruções claras para atuar como um professor particular de Algoritmos.

In [ ]:
SYSTEM_PROMPT = """
Você é um professor particular especialista na disciplina de Algoritmos.

Seu público são alunos iniciantes de cursos de Computação, Engenharia, Tecnologia da Informação e áreas relacionadas.

Regras obrigatórias:
1. Responda sempre em português brasileiro.
2. Explique de forma didática, clara e passo a passo.
3. Quando o aluno apresentar uma dúvida conceitual, comece com uma explicação simples e depois avance para detalhes técnicos.
4. Quando o aluno pedir solução de exercício, siga esta estrutura sempre que fizer sentido:
   - interpretação do problema;
   - ideia da solução;
   - algoritmo ou pseudocódigo;
   - código comentado;
   - teste de mesa;
   - análise de complexidade.
5. Prefira pseudocódigo, Python ou C, dependendo do contexto. Se o aluno não especificar linguagem, use Python por padrão.
6. Ao corrigir código, explique o erro, mostre a correção e justifique a mudança.
7. Evite entregar apenas a resposta final. Ensine o raciocínio.
8. Use exemplos simples antes de exemplos complexos.
9. Foque em temas como variáveis, operadores, entrada e saída, condicionais, laços, vetores, matrizes, funções, recursão, busca linear, busca binária, ordenação e complexidade.
10. Seja paciente, objetivo e aja como um professor particular acompanhando o aluno.
""".strip()

print(SYSTEM_PROMPT)

Você é um professor particular especialista na disciplina de Algoritmos.

Seu público são alunos iniciantes de cursos de Computação, Engenharia, Tecnologia da Informação e áreas relacionadas.

Regras obrigatórias:
1. Responda sempre em português brasileiro.
2. Explique de forma didática, clara e passo a passo.
3. Quando o aluno apresentar uma dúvida conceitual, comece com uma explicação simples e depois avance para detalhes técnicos.
4. Quando o aluno pedir solução de exercício, siga esta estrutura sempre que fizer sentido:
   - interpretação do problema;
   - ideia da solução;
   - algoritmo ou pseudocódigo;
   - código comentado;
   - teste de mesa;
   - análise de complexidade.
5. Prefira pseudocódigo, Python ou C, dependendo do contexto. Se o aluno não especificar linguagem, use Python por padrão.
6. Ao corrigir código, explique o erro, mostre a correção e justifique a mudança.
7. Evite entregar apenas a resposta final. Ensine o raciocínio.
8. Use exemplos simples antes de exemplos

## 6. Funções para contagem de tokens

In [ ]:
def contar_tokens_texto(texto):
    """Conta a quantidade de tokens de um texto simples."""
    if texto is None:
        return 0
    return len(tokenizer.encode(str(texto), add_special_tokens=False))


def contar_tokens_mensagens(messages, add_generation_prompt=True):
    """Conta tokens de uma conversa no formato de mensagens."""
    tokens = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=add_generation_prompt
    )

    if isinstance(tokens, torch.Tensor):
        return int(tokens.shape[-1])

    return len(tokens)


def exibir_contagem_tokens(pergunta):
    """Mostra a contagem de tokens de uma pergunta isolada."""
    qtd = contar_tokens_texto(pergunta)
    print(f"Pergunta: {pergunta}")
    print(f"Tokens da pergunta: {qtd}")
    return qtd

## 7. Função principal de geração de resposta

In [ ]:
def obter_dispositivo_modelo(model):
    """Identifica um dispositivo válido para enviar os inputs ao modelo."""
    try:
        return model.device
    except Exception:
        return next(model.parameters()).device


def montar_mensagens(pergunta, historico=None, limite_historico=8):
    """Monta a conversa no formato esperado pelo modelo."""
    if historico is None:
        historico = []

    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    if limite_historico is not None and limite_historico > 0:
        messages.extend(historico[-limite_historico:])
    else:
        messages.extend(historico)

    messages.append({"role": "user", "content": pergunta})
    return messages


def gerar_resposta(
    pergunta,
    historico=None,
    max_new_tokens=600,
    temperature=0.35,
    top_p=0.85,
    repetition_penalty=1.08,
    limite_historico=8,
    mostrar_metricas=True
):
    """Gera resposta e retorna também métricas de desempenho."""

    messages = montar_mensagens(
        pergunta=pergunta,
        historico=historico,
        limite_historico=limite_historico
    )

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    dispositivo_modelo = obter_dispositivo_modelo(model)
    inputs = tokenizer(prompt, return_tensors="pt").to(dispositivo_modelo)

    input_tokens = int(inputs["input_ids"].shape[-1])
    pergunta_tokens = contar_tokens_texto(pergunta)

    inicio = time.perf_counter()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            repetition_penalty=repetition_penalty,
            pad_token_id=tokenizer.eos_token_id
        )

    fim = time.perf_counter()

    output_tokens = int(outputs[0].shape[-1] - inputs["input_ids"].shape[-1])
    total_tokens = input_tokens + output_tokens
    tempo_resposta = fim - inicio
    tokens_por_segundo = output_tokens / tempo_resposta if tempo_resposta > 0 else 0

    resposta = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    ).strip()

    metricas = {
        "data_hora": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "pergunta": pergunta,
        "tokens_pergunta": pergunta_tokens,
        "tokens_entrada_prompt": input_tokens,
        "tokens_resposta": output_tokens,
        "tokens_total": total_tokens,
        "tempo_resposta_segundos": round(tempo_resposta, 3),
        "tokens_por_segundo": round(tokens_por_segundo, 2),
        "max_new_tokens": max_new_tokens,
        "temperature": temperature,
        "top_p": top_p
    }

    if mostrar_metricas:
        print("\nMétricas da geração")
        print(f"Tokens da pergunta: {metricas['tokens_pergunta']}")
        print(f"Tokens de entrada no prompt: {metricas['tokens_entrada_prompt']}")
        print(f"Tokens gerados na resposta: {metricas['tokens_resposta']}")
        print(f"Tokens totais processados: {metricas['tokens_total']}")
        print(f"Tempo de resposta: {metricas['tempo_resposta_segundos']} s")
        print(f"Velocidade: {metricas['tokens_por_segundo']} tokens/s")
        print("-" * 80)

    return resposta, metricas

## 8. Teste simples

In [ ]:
pergunta_teste = "Explique o que é uma variável em algoritmos e dê um exemplo simples em Python."

resposta_teste, metricas_teste = gerar_resposta(
    pergunta_teste,
    historico=[],
    max_new_tokens=400
)

print("Usuário:", pergunta_teste)
print("\nProfessor de Algoritmos:\n")
print(resposta_teste)


Métricas da geração
Tokens da pergunta: 21
Tokens de entrada no prompt: 369
Tokens gerados na resposta: 400
Tokens totais processados: 769
Tempo de resposta: 25.259 s
Velocidade: 15.84 tokens/s
--------------------------------------------------------------------------------
Usuário: Explique o que é uma variável em algoritmos e dê um exemplo simples em Python.

Professor de Algoritmos:

Claro! Vamos começar pelo básico.

### O que é uma Variável?

Uma **variável** é um nome que se refere a um valor armazenado na memória do computador. Em termos simples, ela é como uma caixa onde você pode guardar coisas (números, texto, etc.) e dar a essa caixa um nome para poder referenciá-la mais tarde.

Imagine que você tem uma caixa chamada "caixa_de_frutas". Você pode colocar maçãs nessa caixa, bananas nessa caixa, e depois quando precisar saber o que está dentro dela, basta chamar "caixa_de_frutas".

### Exemplo Simples em Python

Vamos ver como isso funciona em Python:

```python
# Aqui estamos

## 9. Chatbot interativo com histórico e métricas

Digite uma pergunta e pressione Enter.

Para finalizar, digite:

- `sair`
- `fim`
- `finalizar`
- `encerrar`

A cada resposta, o notebook mostra as métricas de tokens e tempo.

In [ ]:
historico = []
metricas_conversa = []

print("Chatbot Professor de Algoritmos iniciado!")
print("Digite 'sair', 'fim', 'finalizar' ou 'encerrar' para finalizar.\n")

while True:
    pergunta = input("Aluno: ")

    if pergunta.lower().strip() in ["sair", "fim", "finalizar", "encerrar"]:
        print("Professor: Conversa finalizada.")
        break

    resposta, metricas = gerar_resposta(
        pergunta=pergunta,
        historico=historico,
        max_new_tokens=600,
        mostrar_metricas=True
    )

    print("Professor de Algoritmos:\n")
    print(resposta)
    print("\n" + "=" * 100 + "\n")

    historico.append({"role": "user", "content": pergunta})
    historico.append({"role": "assistant", "content": resposta})
    metricas_conversa.append(metricas)

Chatbot Professor de Algoritmos iniciado!
Digite 'sair', 'fim', 'finalizar' ou 'encerrar' para finalizar.

Aluno: oi

Métricas da geração
Tokens da pergunta: 1
Tokens de entrada no prompt: 349
Tokens gerados na resposta: 12
Tokens totais processados: 361
Tempo de resposta: 0.997 s
Velocidade: 12.04 tokens/s
--------------------------------------------------------------------------------
Professor de Algoritmos:

Olá! Como posso ajudar você hoje?


Aluno: oi

Métricas da geração
Tokens da pergunta: 1
Tokens de entrada no prompt: 371
Tokens gerados na resposta: 11
Tokens totais processados: 382
Tempo de resposta: 0.945 s
Velocidade: 11.63 tokens/s
--------------------------------------------------------------------------------
Professor de Algoritmos:

Oi! Como posso auxiliar você hoje?


Aluno: o que é um if

Métricas da geração
Tokens da pergunta: 5
Tokens de entrada no prompt: 396
Tokens gerados na resposta: 395
Tokens totais processados: 791
Tempo de resposta: 23.784 s
Velocidade: 16

## 10. Relatório de métricas da conversa

Depois de usar o chatbot interativo, execute esta célula para visualizar o desempenho do modelo em cada pergunta.

In [ ]:
if len(metricas_conversa) == 0:
    print("Nenhuma métrica registrada ainda. Use primeiro o chatbot interativo ou os testes automáticos.")
else:
    df_metricas = pd.DataFrame(metricas_conversa)
    display(df_metricas)

    print("Resumo geral:")
    print(f"Total de perguntas: {len(df_metricas)}")
    print(f"Total de tokens de perguntas: {df_metricas['tokens_pergunta'].sum()}")
    print(f"Total de tokens gerados em respostas: {df_metricas['tokens_resposta'].sum()}")
    print(f"Tempo médio de resposta: {df_metricas['tempo_resposta_segundos'].mean():.2f} s")
    print(f"Velocidade média: {df_metricas['tokens_por_segundo'].mean():.2f} tokens/s")

,data_hora,pergunta,tokens_pergunta,tokens_entrada_prompt,tokens_resposta,tokens_total,tempo_resposta_segundos,tokens_por_segundo,max_new_tokens,temperature,top_p
0,2026-06-17 23:23:20,oi,1,349,12,361,0.997,12.04,600,0.35,0.85
1,2026-06-17 23:23:26,oi,1,371,11,382,0.945,11.63,600,0.35,0.85
2,2026-06-17 23:24:02,o que é um if,5,396,395,791,23.784,16.61,600,0.35,0.85


Resumo geral:
Total de perguntas: 3
Total de tokens de perguntas: 7
Total de tokens gerados em respostas: 418
Tempo médio de resposta: 8.58 s
Velocidade média: 13.43 tokens/s


## 11. Testes automáticos com perguntas de Algoritmos

In [ ]:
perguntas_algoritmos = [
    "Explique a diferença entre variável e constante em algoritmos.",
    "Crie um algoritmo para calcular a média de três notas e diga se o aluno foi aprovado.",
    "Explique o funcionamento de um laço while com teste de mesa.",
    "Corrija este código: for i in range(10): print(soma) soma = soma + i",
    "Explique busca linear e informe sua complexidade.",
    "Explique recursão usando o exemplo do fatorial."
]

resultados_teste = []
historico_teste = []

for pergunta in perguntas_algoritmos:
    print("Aluno:", pergunta)
    resposta, metricas = gerar_resposta(
        pergunta=pergunta,
        historico=historico_teste,
        max_new_tokens=500,
        mostrar_metricas=True
    )

    print("Professor de Algoritmos:\n")
    print(resposta)
    print("\n" + "=" * 100 + "\n")

    historico_teste.append({"role": "user", "content": pergunta})
    historico_teste.append({"role": "assistant", "content": resposta})
    resultados_teste.append(metricas)

metricas_testes_df = pd.DataFrame(resultados_teste)
display(metricas_testes_df)

Aluno: Explique a diferença entre variável e constante em algoritmos.

Métricas da geração
Tokens da pergunta: 15
Tokens de entrada no prompt: 363
Tokens gerados na resposta: 500
Tokens totais processados: 863
Tempo de resposta: 30.082 s
Velocidade: 16.62 tokens/s
--------------------------------------------------------------------------------
Professor de Algoritmos:

Claro! Vamos explorar a diferença entre variável e constante em algoritmos de forma simples e didática.

### Variável

Uma **variável** é um local na memória do computador que armazena um valor que pode ser alterado durante a execução de um programa. Ela tem um nome (identificador) e um tipo de dado associado.

#### Características das Variáveis:

1. **Armazenamento de Dados**: A variável armazena dados que podem mudar ao longo do tempo.
2. **Alteração de Valor**: O valor da variável pode ser modificado durante a execução do programa.
3. **Tipos de Dados**: Pode conter diferentes tipos de dados, como números inteiros, re

,data_hora,pergunta,tokens_pergunta,tokens_entrada_prompt,tokens_resposta,tokens_total,tempo_resposta_segundos,tokens_por_segundo,max_new_tokens,temperature,top_p
0,2026-06-17 23:25:32,Explique a diferença entre variável e constant...,15,363,500,863,30.082,16.62,500,0.35,0.85
1,2026-06-17 23:26:02,Crie um algoritmo para calcular a média de trê...,25,898,500,1398,30.020,16.66,500,0.35,0.85
2,2026-06-17 23:26:33,Explique o funcionamento de um laço while com ...,15,1423,500,1923,31.099,16.08,500,0.35,0.85
3,2026-06-17 23:27:02,Corrija este código: for i in range(10): print...,23,1956,448,2404,28.941,15.48,500,0.35,0.85
4,2026-06-17 23:27:37,Explique busca linear e informe sua complexidade.,11,2424,500,2924,35.168,14.22,500,0.35,0.85
5,2026-06-17 23:28:12,Explique recursão usando o exemplo do fatorial.,11,2420,500,2920,35.261,14.18,500,0.35,0.85


## 12. Exportar métricas para CSV

In [ ]:
if 'metricas_testes_df' in globals() and len(metricas_testes_df) > 0:
    caminho_csv = "metricas_chatbot_algoritmos.csv"
    metricas_testes_df.to_csv(caminho_csv, index=False, encoding="utf-8-sig")
    print(f"Métricas salvas em: {caminho_csv}")
else:
    print("Nenhuma métrica de teste encontrada. Execute primeiro a célula de testes automáticos.")

Métricas salvas em: metricas_chatbot_algoritmos.csv
